# m36 lab · Evidence in numbers

*Open EP Mapping Lab. Education only: not for diagnosis, treatment or any clinical decision. Not legal or regulatory
advice.*

| Part | What you do | Data |
| --- | --- | --- |
| A | Zero-failure and c-failure sample sizes, checked by simulation | synthetic |
| B | Clopper–Pearson bounds and their coverage, against the normal approximation | synthetic |
| C | Series reliability by Monte Carlo, and what a redundant pair buys | synthetic |
| D | A traceability matrix from requirements to this course's own tests | this repository |
| E | A small risk register: estimate, control, and check residual risk | synthetic |

**How to run.** From the course repository: `pip install -r requirements.txt`, then open this notebook in Jupyter.
In Google Colab the first code cell fetches the course code for you. The whole lab runs in under a minute.

Cells that start with `assert` are **checks**: they pass with the code as given. If you change something and a
check fails, that is information, not an error to hide. All data here are synthetic or come from the course's own
models; nothing describes a real product or patient.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# Find the shared model code in labs/python/epml. In Colab, fetch the course repository first.
try:
    import epml
except ImportError:
    if "google.colab" in sys.modules:
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/open-ep-mapping-lab/open-ep-mapping-lab.github.io",
                        "open-ep-mapping-lab"], check=True)
        sys.path.insert(0, "open-ep-mapping-lab/labs/python")
    else:
        sys.path.insert(0, str(Path.cwd() / "labs" / "python"))
    import epml

plt.rcParams.update({"figure.figsize": (8, 3.2), "axes.grid": True, "axes.spines.top": False, "axes.spines.right": False})

import ast
import itertools

from scipy import stats

from epml import product as pr

## Part A · How many units?

For each target reliability $R$, confidence $C$ and number of allowed failures $c$, the module's plan is the smallest
$n$ with $P(\le c \text{ failures} \mid R) \le 1 - C$. The simulation tests many lots that are exactly as good as the
target: at most a share $1 - C$ of them may pass.

In [ ]:
rng = np.random.default_rng(36)
print(" R     C    c    n   pass rate at R (simulated)")
table = []
for R, C, c in itertools.product([0.90, 0.95, 0.99], [0.90, 0.95], [0, 1, 2]):
    n = pr.acceptance_n(R, C, c)
    fails = rng.binomial(n, 1 - R, 100_000)
    passed = np.mean(fails <= c)
    table.append((R, C, c, n, passed))
    print(f"{R:.2f}  {C:.2f}  {c}  {n:4d}   {passed:.3f}")

In [ ]:
assert all(passed <= 1 - C + 0.005 for R, C, c, n, passed in table)
assert all(n == pr.success_run_n(R, C) for R, C, c, n, _ in table if c == 0)
assert pr.acceptance_n(0.95, 0.95, 0) == 59 and pr.acceptance_n(0.99, 0.95, 0) == 299

## Part B · How sure is a pass rate?

The exact Clopper–Pearson interval covers the true rate at least 95% of the time. The common normal approximation
$\hat p \pm 1.96\sqrt{\hat p(1-\hat p)/n}$ can fall short when the rate is close to 1, which is exactly where
verification results usually are.

In [ ]:
def coverage(p, n, method, reps=20_000, seed=0):
    xs = np.random.default_rng(seed).binomial(n, p, reps)
    hits = 0
    for x in xs:
        if method == "exact":
            lo, hi = pr.clopper_pearson(x, n)
        else:
            ph = x / n
            half = 1.96 * np.sqrt(ph * (1 - ph) / n)
            lo, hi = ph - half, ph + half
        hits += lo <= p <= hi
    return hits / reps

for p in (0.90, 0.97, 0.99):
    print(f"p = {p}: exact {coverage(p, 59, 'exact'):.3f}, normal approximation {coverage(p, 59, 'normal'):.3f}")

In [ ]:
assert all(coverage(p, 59, "exact") >= 0.945 for p in (0.90, 0.97, 0.99))
assert coverage(0.99, 59, "normal") < 0.90       # the approximation fails near 1

## Part C · Reliability in series, and a redundant pair

Three parts in series with constant failure rates (illustrative values). Then the weakest part is doubled in parallel:
the pair fails only if both copies fail, so its reliability is $1 - (1 - R)^2$.

In [ ]:
rates = np.array([1e-4, 5e-5, 2e-4])        # per hour, illustrative
t_m = 2000.0                                # mission time (h)
life = rng.exponential(1 / rates, size=(400_000, 3))
series_sim = np.mean(life.min(axis=1) > t_m)
series_formula = float(pr.reliability(t_m, pr.series_failure_rate(rates)))

weak = int(np.argmax(rates))
backup = rng.exponential(1 / rates[weak], size=life.shape[0])
life_red = life.copy()
life_red[:, weak] = np.maximum(life[:, weak], backup)          # the pair survives while either copy does
red_sim = np.mean(life_red.min(axis=1) > t_m)
r_parts = np.exp(-rates * t_m)
r_parts_red = r_parts.copy()
r_parts_red[weak] = 1 - (1 - r_parts[weak]) ** 2
red_formula = float(np.prod(r_parts_red))
print(f"series: simulated {series_sim:.4f}, formula {series_formula:.4f}; "
      f"with a redundant part {weak}: simulated {red_sim:.4f}, formula {red_formula:.4f}")

In [ ]:
assert abs(series_sim - series_formula) < 3e-3
assert abs(red_sim - red_formula) < 3e-3
assert red_formula > series_formula

## Part D · Traceability: requirements to tests

The course's own rules (CLAUDE.md) are requirements, and its `tests/` folder holds the verification. Below, each
requirement names the tests that verify it. The code reads the test files and reports any requirement whose tests do
not exist, and any requirement that is verified some other way. Two are verified by people or by the CI setup, not
by a test, and that is fine as long as it is written down.

In [ ]:
ROOT = next(p for p in [Path.cwd().parents[1], Path.cwd(), Path("open-ep-mapping-lab")] if (p / "tests").is_dir())

requirements = {
    "REQ-01 Every module has the template sections in order": ["test_required_sections_in_order"],
    "REQ-02 Every demo has one badge and an Assumptions list": ["test_demos_have_badge_and_assumptions"],
    "REQ-03 Every numbered equation has a test": ["test_every_equation_has_a_test"],
    "REQ-04 Every citation resolves to a checked reference": ["test_citations_are_in_references_bib",
                                                              "test_references_have_doi_or_pmid"],
    "REQ-05 Questions have answers": ["test_questions_have_answers"],
    "REQ-06 Demo math matches the tested model": ["test_m34_demo_math", "test_m35_demo_math", "test_m36_demo_math"],
    "REQ-07 A code lab runs in CI in under 5 minutes": "CI: timeout-minutes in .github/workflows/ci.yml",
    "REQ-08 A human records expert review": "Manual: review_record.json, signed by a person",
}

tests = set()
for f in (ROOT / "tests").glob("test_*.py"):
    tree = ast.parse(f.read_text())
    tests |= {n.name for n in ast.walk(tree) if isinstance(n, ast.FunctionDef) and n.name.startswith("test_")}

matrix = {}
for req, how in requirements.items():
    if isinstance(how, str):
        matrix[req] = ("other", how)
    else:
        missing = [t for t in how if t not in tests]
        matrix[req] = ("MISSING " + ", ".join(missing), how) if missing else ("tested", how)
for req, (status, how) in matrix.items():
    print(f"{status:10s} {req}")

In [ ]:
assert not [r for r, (s, _) in matrix.items() if s.startswith("MISSING")]
assert sum(s == "other" for s, _ in matrix.values()) == 2
assert "timeout-minutes" in (ROOT / ".github" / "workflows" / "ci.yml").read_text()

## Part E · A small risk register

Each line is a generic, illustrative hazardous situation for a mapping system. $P_1$ is the probability that the
hazardous situation occurs, $P_2$ that it leads to harm, and severity is on a 1–5 scale. ISO 14971 asks each
manufacturer to define its own acceptability criteria; the matrix below is a made-up example, which you are invited
to change. A risk control lowers $P_1$ or $P_2$; the residual risk is checked again.

In [ ]:
register = [
    # name, P1, P2, severity
    ("displayed signal belongs to another channel", 2e-3, 0.1, 3),
    ("catheter position shown late", 5e-2, 0.05, 2),
    ("map built from points of two different rhythms", 5e-3, 0.2, 3),
]

def acceptable(p_harm, severity):
    """Example criterion: the more severe the harm, the smaller its acceptable probability."""
    limit = {1: 1e-2, 2: 1e-3, 3: 1e-4, 4: 1e-5, 5: 1e-6}[severity]
    return p_harm <= limit

controls = {   # factor applied to P1 by a risk control (illustrative)
    "displayed signal belongs to another channel": 0.05,
    "catheter position shown late": 0.1,
    "map built from points of two different rhythms": 0.05,
}
for name, p1, p2, sev in register:
    before = pr.harm_probability(p1, p2)
    after = pr.harm_probability(p1 * controls[name], p2)
    print(f"{name:48s} P = {before:.1e} -> {after:.1e}  acceptable: {acceptable(before, sev)} -> {acceptable(after, sev)}")

In [ ]:
assert all(not acceptable(pr.harm_probability(p1, p2), s) for _, p1, p2, s in register)          # all need control
assert all(acceptable(pr.harm_probability(p1 * controls[n], p2), s) for n, p1, p2, s in register)  # all residual OK